In [2]:
import cv2
import numpy as np
import mediapipe as mp
import time
import json

import pyrealsense2 as rs  # Intel RealSense SDK


# --------------------------
# 1. 빨간 마커 탐지 (HSV 마스크)
# --------------------------
def find_red_markers(frame_bgr):
    """
    BGR 이미지를 입력받아,
    HSV 마스크로 빨간색 영역을 찾아
    각 덩어리(마커)의 중심 좌표 리스트를 반환.
    """
    hsv = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2HSV)

    # 빨간색 범위 (조금 조인 버전 – 필요하면 다시 튜닝)
    lower_red1 = np.array([0,   150, 120])
    upper_red1 = np.array([8,   255, 255])
    lower_red2 = np.array([172, 150, 120])
    upper_red2 = np.array([180, 255, 255])

    mask1 = cv2.inRange(hsv, lower_red1, upper_red1)
    mask2 = cv2.inRange(hsv, lower_red2, upper_red2)
    mask = cv2.bitwise_or(mask1, mask2)

    # 잡음 제거용 morphological operation
    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)

    # 컨투어(빨간 영역) 찾기
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    red_points = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < 30:   # 마커 크기에 따라 20~50 사이에서 조절
            continue
        M = cv2.moments(cnt)
        if M["m00"] == 0:
            continue
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])
        red_points.append((cx, cy))

    return red_points, mask


def in_box(pt, box):
    """pt = (x, y), box = (x_min, y_min, x_max, y_max)"""
    x, y = pt
    x_min, y_min, x_max, y_max = box
    return (x_min <= x <= x_max) and (y_min <= y <= y_max)


# --------------------------
# 2. 메인 함수 (D435 사용)
# --------------------------
def main():
    # ==============
    # RealSense 초기화
    # ==============
    pipeline = rs.pipeline()
    config = rs.config()

    # 컬러 + 뎁스 스트림 활성화
    config.enable_stream(rs.stream.color, 640, 480, rs.format.bgr8, 30)
    config.enable_stream(rs.stream.depth, 640, 480, rs.format.z16, 30)

    # 파이프라인 시작
    profile = pipeline.start(config)

    # depth를 color에 align (픽셀 좌표 맞추기)
    align_to = rs.stream.color
    align = rs.align(align_to)

    # ==============
    # MediaPipe FaceMesh 초기화
    # ==============
    mp_face_mesh = mp.solutions.face_mesh
    face_mesh = mp_face_mesh.FaceMesh(
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
    )

    # 왼/오 볼 윤곽을 이루는 랜드마크 인덱스
    left_cheek_region =  [234,  93, 132,  58, 172, 136, 150, 176, 148, 152]
    right_cheek_region = [454, 323, 361, 288, 397, 365, 379, 400, 377, 152]

    # 볼 영역 bounding box 확장 margin (픽셀 단위)
    margin_x = 15
    margin_y = 15

    try:
        while True:
            # ==============
            # 1) D435에서 프레임 받기
            # ==============
            frames = pipeline.wait_for_frames()
            aligned_frames = align.process(frames)

            color_frame = aligned_frames.get_color_frame()
            depth_frame = aligned_frames.get_depth_frame()

            if not color_frame or not depth_frame:
                continue

            # numpy 배열로 변환
            frame = np.asanyarray(color_frame.get_data())  # BGR

            h, w, _ = frame.shape

            # ==============
            # 2) FaceMesh로 볼 polygon 계산
            # ==============
            rgb_image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = face_mesh.process(rgb_image)

            left_box = None
            right_box = None

            if results.multi_face_landmarks:
                face_landmarks = results.multi_face_landmarks[0]

                # --- 왼쪽 볼 ---
                left_poly = []
                for idx in left_cheek_region:
                    lm = face_landmarks.landmark[idx]
                    u = int(lm.x * w)
                    v = int(lm.y * h)
                    left_poly.append((u, v))
                    cv2.circle(frame, (u, v), 3, (0, 0, 255), -1)  # 빨간 점

                if len(left_poly) >= 3:
                    left_poly_np = np.array(left_poly, np.int32)
                    cv2.polylines(frame, [left_poly_np], True, (0, 0, 255), 1)

                    lx_min = max(min(p[0] for p in left_poly) - margin_x, 0)
                    lx_max = min(max(p[0] for p in left_poly) + margin_x, w - 1)
                    ly_min = max(min(p[1] for p in left_poly) - margin_y, 0)
                    ly_max = min(max(p[1] for p in left_poly) + margin_y, h - 1)
                    left_box = (lx_min, ly_min, lx_max, ly_max)

                    cv2.rectangle(frame, (lx_min, ly_min), (lx_max, ly_max), (0, 0, 150), 1)

                # --- 오른쪽 볼 ---
                right_poly = []
                for idx in right_cheek_region:
                    lm = face_landmarks.landmark[idx]
                    u = int(lm.x * w)
                    v = int(lm.y * h)
                    right_poly.append((u, v))
                    cv2.circle(frame, (u, v), 3, (255, 0, 0), -1)  # 파란 점

                if len(right_poly) >= 3:
                    right_poly_np = np.array(right_poly, np.int32)
                    cv2.polylines(frame, [right_poly_np], True, (255, 0, 0), 1)

                    rx_min = max(min(p[0] for p in right_poly) - margin_x, 0)
                    rx_max = min(max(p[0] for p in right_poly) + margin_x, w - 1)
                    ry_min = max(min(p[1] for p in right_poly) - margin_y, 0)
                    ry_max = min(max(p[1] for p in right_poly) + margin_y, h - 1)
                    right_box = (rx_min, ry_min, rx_max, ry_max)

                    cv2.rectangle(frame, (rx_min, ry_min), (rx_max, ry_max), (150, 0, 0), 1)

            # ==============
            # 3) HSV로 빨간 마커 탐지
            # ==============
            red_points, red_mask = find_red_markers(frame)

            left_markers = []
            right_markers = []
            others = []

            # ==============
            # 4) 마커를 L/R/others로 분류 + depth 측정
            # ==============
            for (cx, cy) in red_points:
                in_left = left_box is not None and in_box((cx, cy), left_box)
                in_right = right_box is not None and in_box((cx, cy), right_box)

                # RealSense depth: (x,y) 픽셀의 거리(m)
                depth_m = float(depth_frame.get_distance(cx, cy))  # meter 단위

                if in_left and not in_right:
                    left_markers.append((cx, cy, depth_m))
                    # 왼쪽 볼 위의 마커: 연두색 + depth 표시
                    cv2.circle(frame, (cx, cy), 7, (0, 255, 0), -1)
                    cv2.putText(
                        frame,
                        f"L {depth_m:.3f}m",
                        (cx + 5, cy - 5),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.4,
                        (0, 255, 0),
                        1,
                        cv2.LINE_AA,
                    )
                elif in_right and not in_left:
                    right_markers.append((cx, cy, depth_m))
                    # 오른쪽 볼 위의 마커: 하늘색 + depth 표시
                    cv2.circle(frame, (cx, cy), 7, (255, 255, 0), -1)
                    cv2.putText(
                        frame,
                        f"R {depth_m:.3f}m",
                        (cx + 5, cy - 5),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.4,
                        (255, 255, 0),
                        1,
                        cv2.LINE_AA,
                    )
                else:
                    others.append((cx, cy, depth_m))
                    # 그 외: 보라색
                    cv2.circle(frame, (cx, cy), 6, (255, 0, 255), -1)

            # ==============
            # 5) JSON 로깅 (좌표 + depth)
            # ==============
            payload = {
                "timestamp": time.time(),
                "left_markers": [
                    {"u": int(u), "v": int(v), "depth_m": float(d)}
                    for (u, v, d) in left_markers
                ],
            }
            payload["right_markers"] = [
                {"u": int(u), "v": int(v), "depth_m": float(d)}
                for (u, v, d) in right_markers
            ]
            payload["others"] = [
                {"u": int(u), "v": int(v), "depth_m": float(d)}
                for (u, v, d) in others
            ]

            print(json.dumps(payload), flush=True)

            # ==============
            # 6) 화면 표시
            # ==============
            cv2.imshow("D435 Color + Cheek + Red Markers", frame)
            # 마스크 확인하고 싶으면 주석 해제
            # cv2.imshow("Red Mask", red_mask)

            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"):
                break

    finally:
        face_mesh.close()
        pipeline.stop()
        cv2.destroyAllWindows()


if __name__ == "__main__":
    main()


{"timestamp": 1764129777.9544773, "left_markers": [], "right_markers": [], "others": []}
{"timestamp": 1764129777.9740648, "left_markers": [], "right_markers": [], "others": []}
{"timestamp": 1764129778.0105386, "left_markers": [], "right_markers": [], "others": []}
{"timestamp": 1764129778.0269005, "left_markers": [], "right_markers": [], "others": []}
{"timestamp": 1764129778.0900264, "left_markers": [], "right_markers": [], "others": []}
{"timestamp": 1764129778.1246192, "left_markers": [], "right_markers": [], "others": []}
{"timestamp": 1764129778.1449442, "left_markers": [], "right_markers": [], "others": []}
{"timestamp": 1764129778.161973, "left_markers": [], "right_markers": [], "others": []}
{"timestamp": 1764129778.1945205, "left_markers": [], "right_markers": [], "others": []}
{"timestamp": 1764129778.228125, "left_markers": [], "right_markers": [], "others": []}
{"timestamp": 1764129778.256514, "left_markers": [], "right_markers": [], "others": []}
{"timestamp": 1764129778

KeyboardInterrupt: 